# Arquitectura de redes neuronales para el perfilamiento delictivo territorial

**Proyecto:** análisis de incidentes delictivos de Los Ángeles (LAPD), registros históricos LAPD (periodo determinado por el archivo).

**Objetivo.** Construir un flujo reproducible que (1) caracterice cada zona por su **mezcla de seis perfiles delictivos consolidados** (P01–P06), (2) identifique el perfil principal, los secundarios y la prioridad de cada zona, y (3) entrene una red neuronal multicapa (MLP) para pronosticar los incidentes del mes siguiente por **zona × perfil delictivo × mes**. El modelo se compara con referencias clásicas (persistencia, gradient boosting y random forest) usando métricas de error honestas.

**Preguntas de análisis.**

1. ¿Qué combinación de perfiles y submodalidades define la composición delictiva de cada zona?
2. ¿Qué perfil es el principal y cuáles son secundarios en cada zona, con qué evidencia y qué nivel de mezcla?
3. ¿Qué perfil debe priorizarse cuando se consideran frecuencia, sobrerrepresentación, tendencia y severidad, sin confundir prioridad con delito más frecuente?
4. ¿Qué perfiles concentran los mayores errores y los mayores incrementos previstos?
5. ¿La MLP supera a la persistencia y a los modelos de árboles en el pronóstico mensual?

**Unidad de análisis.** Zona de reporte (`zone`) × perfil delictivo (P01–P06) × mes. El target es el número de incidentes del mes siguiente. No se modela riesgo individual ni probabilidad de victimización.

**Advertencia.** Los datos reflejan denuncias registradas, no la criminalidad real. Un aumento puede deberse a mayor registro, cambios de política o estacionalidad. El notebook es analítico y no causal.

## 1. Enfoque: perfil temático, composición y prioridad

El trabajo distingue dos objetos que no deben confundirse:

- **Perfil delictivo individual (P01–P06).** Es una categoría temática construida a partir de las 138 tipificaciones de `crime_description`. Cada zona contiene **todos** los perfiles en distintas proporciones; el perfil principal es el de mayor peso, no el único delito de la zona. Los perfiles pueden coexistir y tienen submodalidades (por ejemplo, vehículo robado vs. robo de objetos dentro del vehículo).
- **Prioridad.** Es una decisión normativa y multi-criterio (frecuencia, sobrerrepresentación, tendencia, severidad y evidencia). No se deriva automáticamente del perfil más frecuente.

La predicción supervisada usa como target el conteo por **zona × perfil × mes**. La MLP aprende del historial reciente, estacionalidad, volumen y perfil. No se agrupan territorios por similitud estadística: el objetivo no es descubrir tipos de zonas, sino describir la composición delictiva y priorizar perfiles.

**Criterio de interpretación:** describir no es predecir. Un perfil puede ser útil para caracterizar y priorizar una zona aunque no mejore el MAE del pronóstico.

## 2. Justificación, supuestos y límites

**¿Por qué una red neuronal?** La relación entre rezagos, estacionalidad y perfil delictivo puede ser no lineal y contener interacciones (por ejemplo, una zona con mezcla residencial puede reaccionar distinto a la temporada que una zona comercial). Una MLP es una aproximación flexible y compacta para ese tipo de superficie. No se afirma superioridad teórica: la comparación empírica decide.

**¿Por qué no CNN/RNN?** La serie por zona-perfil es corta (decenas de meses), irregular y con muchos ceros. Una CNN o una LSTM requeriría más historia y una reconstrucción de secuencias más frágil. La formulación tabular con rezagos explícitos es más estable y auditable para este tamaño de datos.

**Supuestos.**

- Las 138 tipificaciones se pueden mapear de forma reproducible a seis perfiles operativos; el mapeo no es una taxonomía jurídica universal.
- El perfil delictivo y la zona son comparables a lo largo del periodo.
- El mes calendario captura parte de la estacionalidad; no se modelan eventos extraordinarios.
- El número de denuncias es un proxy del volumen delictivo, no una tasa poblacional.
- La composición de perfiles de una zona es razonablemente estable en el horizonte de un mes.

**Límites.**

- No hay población por zona: no se calculan tasas por habitante.
- 2023-12 se excluye por estar incompleto.
- Las zonas con muy pocos incidentes se excluyen para estabilizar proporciones.
- Las tipificaciones genéricas o ambiguas se conservan como `NO_ASIGNADO` y se reportan aparte; no se crea un séptimo perfil sustantivo.
- El análisis no identifica causalidad ni permite inferir responsabilidad individual.
- La prioridad incorpora pesos de severidad explícitos y discutibles; debe leerse como herramienta de apoyo, no como sentencia.
- Los resultados dependen del periodo y de la calidad del registro; deben reentrenarse si cambia el esquema de categorías.

**Control de fuga temporal:** los perfiles se ajustan una sola vez con un periodo de calibración anterior a TODOS los meses objetivo usados para entrenar, validar y probar. Son una fotografía histórica, no etiquetas recalculadas con información futura.

## 3. Importaciones y configuración reproducible

Se fija una semilla para NumPy, Python y TensorFlow. Los paths son relativos o detectados automáticamente para funcionar en Colab y en JupyterLab.

In [ ]:
# Las dependencias se instalan con server/requirements.txt antes del arranque.
import importlib.util
if importlib.util.find_spec("tensorflow") is None:
    raise RuntimeError("Falta TensorFlow. Instala server/requirements.txt en el entorno del backend.")


In [ ]:
import os
import gc
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("TensorFlow:", tf.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)

## 4. Carga del dataset

El archivo esperado es `Crime_Data_from_2020_to_Present-vip.csv` (o `.xlsx`) en el directorio del notebook. La búsqueda es relativa para no depender de rutas absolutas. En Colab, si no se encuentra, se abre el cargador de archivos.

In [ ]:
USECOLS = {
    "date_reported", "date_occurred", "reporting_district", "area_name",
    "crime_description", "premise_description", "weapon_description",
    "modus_operandi", "latitude", "longitude", "victim_age",
    "victim_sex", "victim_descent",
}


def find_data_path():
    # Locate the dataset without absolute paths.
    override = os.environ.get("CRIME_DATA")
    if override:
        p = Path(override)
        if p.exists():
            return p

    candidates = []
    for pattern in ("*Crime*Data*.csv", "*Crime*Data*.xlsx", "*crime*.csv", "*crime*.xlsx"):
        candidates.extend(sorted(Path(".").glob(pattern)))
    if candidates:
        return candidates[0]

    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            return Path(next(iter(uploaded)))
    except Exception:
        pass

    file_name = globals().get("FILE_NAME")
    if file_name and Path(file_name).exists():
        return Path(file_name)

    raise FileNotFoundError(
        "No encontré el CSV. Coloca 'Crime_Data_from_2020_to_Present-vip.csv' "
        "junto al notebook, define FILE_NAME, o usa CRIME_DATA=/ruta/al/archivo."
    )


DATA_PATH = find_data_path()
NROWS = int(os.environ["CRIME_NROWS"]) if os.environ.get("CRIME_NROWS") else None
print("Archivo:", DATA_PATH)
print("Filas máximas:", NROWS if NROWS else "todas")

if DATA_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(DATA_PATH, usecols=lambda c: str(c).strip().lower() in USECOLS, nrows=NROWS)
else:
    df = pd.read_csv(
        DATA_PATH,
        usecols=lambda c: str(c).strip().lower() in USECOLS,
        low_memory=False,
        nrows=NROWS,
    )

df.columns = df.columns.str.strip().str.lower()
required = {"date_occurred", "reporting_district", "crime_description"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(missing)}")

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
display(df.head())

## 5. Auditoría y preparación

Se normalizan textos, se convierten coordenadas y edades, se parsean fechas **día primero** (el CSV usa `DD/MM/YYYY`), se derivan variables temporales y se define la unidad territorial. También se excluye el último mes por estar incompleto.

**Paso central:** se auditan las 138 tipificaciones de `crime_description` y se mapean a seis perfiles consolidados:

- P01 Delitos Vehiculares y Movilidad
- P02 Robo y Hurto Patrimonial
- P03 Robos Residenciales e Intrusión
- P04 Delitos Contra la Vida y la Integridad
- P05 Delitos Sexuales
- P06 Fraude, Vandalismo y Daños

Cada asignación conserva una **submodalidad**, un nivel de confianza y una nota. Las tipificaciones genéricas o inclasificables se marcan como `NO_ASIGNADO`; no se crea un séptimo perfil. La tabla `auditoria_tipificaciones.csv` permite revisar y corregir el mapeo sin reescribir el notebook.

In [ ]:
D = df
del df

# --- 5.1 Texto y tipos -------------------------------------------------------
TEXT_COLS = [
    "crime_description", "premise_description", "weapon_description",
    "modus_operandi", "area_name", "victim_sex", "victim_descent",
    "reporting_district",
]
for col in TEXT_COLS:
    if col not in D:
        D[col] = "NO_REGISTRADO"
    D[col] = (
        D[col].astype("string").str.strip().str.upper()
        .fillna("NO_REGISTRADO")
        .replace({"": "NO_REGISTRADO", "NAN": "NO_REGISTRADO", "NONE": "NO_REGISTRADO"})
    )

for col in ["latitude", "longitude", "victim_age"]:
    D[col] = pd.to_numeric(D[col], errors="coerce") if col in D else np.nan

# Exportaciones LAPD pueden codificar 34.0141 como 340141.
for col, limit in [("latitude", 90), ("longitude", 180)]:
    for _ in range(7):
        mask = D[col].abs() > limit
        if not mask.any():
            break
        D.loc[mask, col] = D.loc[mask, col] / 10

# --- 5.2 Fechas: detectar DD/MM vs MM/DD y parsear explícitamente -----------
def parse_lapd_dates(series):
    s = series.astype("string").str.strip()
    sample = s.dropna().head(50000)
    parts = sample.str.extract(r"^(\d{1,2})[/-](\d{1,2})[/-](\d{4})")
    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")
    dayfirst = bool((first > 12).sum() >= (second > 12).sum()) if len(parts) else True
    if dayfirst:
        formats = ["%d/%m/%Y %H:%M", "%d/%m/%Y", "%d-%m-%Y %H:%M", "%d-%m-%Y"]
    else:
        formats = ["%m/%d/%Y %H:%M", "%m/%d/%Y", "%m-%d-%Y %H:%M", "%m-%d-%Y"]
    formats += ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d"]
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in formats:
        if out.notna().all():
            break
        out = out.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    out = out.fillna(pd.to_datetime(s, errors="coerce", dayfirst=dayfirst))
    return out, dayfirst


D["date_occurred"], dayfirst_occ = parse_lapd_dates(D["date_occurred"])
D["date_reported"], dayfirst_rep = parse_lapd_dates(D["date_reported"])
print("Orden detectado:", "DD/MM/YYYY" if dayfirst_occ else "MM/DD/YYYY")
print(f"Fechas no parseadas -> ocurrido: {D.date_occurred.isna().sum():,} | reportado: {D.date_reported.isna().sum():,}")

D["month"] = D["date_occurred"].dt.to_period("M")
D["month_num"] = D["month"].dt.month
D["year"] = D["month"].dt.year
D["hour"] = D["date_occurred"].dt.hour
D["weekday"] = D["date_occurred"].dt.dayofweek
D["night"] = D["hour"].between(20, 23) | D["hour"].between(0, 5)
D["weekend"] = D["weekday"].isin([5, 6])

# Retraso de reporte: normalizar a día para no confundir la hora con un negativo.
D["report_delay_days"] = (
    D["date_reported"].dt.normalize() - D["date_occurred"].dt.normalize()
).dt.days
D["same_day_report"] = D["report_delay_days"].eq(0)
D["report_delay_negative"] = D["report_delay_days"].lt(0)

D["valid_geo"] = (
    D.latitude.between(-90, 90) & D.longitude.between(-180, 180)
    & ~((D.latitude == 0) & (D.longitude == 0))
)
D["victim_age_valid"] = D.victim_age.where(D.victim_age.between(0, 110))

# --- 5.3 Catálogo de seis perfiles delictivos consolidados -------------------
PROFILE_CODES = ["P01", "P02", "P03", "P04", "P05", "P06"]
PROFILE_LABELS = {
    "P01": "Delitos Vehiculares y Movilidad",
    "P02": "Robo y Hurto Patrimonial",
    "P03": "Robos Residenciales e Intrusión",
    "P04": "Delitos Contra la Vida y la Integridad",
    "P05": "Delitos Sexuales",
    "P06": "Fraude, Vandalismo y Daños",
}
PROFILE_SEVERITY = {"P01": 0.7, "P02": 0.6, "P03": 0.8, "P04": 1.0, "P05": 1.0, "P06": 0.5}

PROFILE_DEFINITIONS = pd.DataFrame([
    {"perfil": "P01", "nombre": PROFILE_LABELS["P01"],
     "descripcion": "Sustracción de vehículos, autopartes, accesorios, objetos dentro de vehículos, bicicletas y movilidad personal; incluye infracciones de movilidad cuando el registro lo permite.",
     "submodalidades": "VEHICULO_ROBADO; ACCESORIOS_VEHICULO; BICICLETA_MOVILIDAD; CONDUCCION",
     "peso_severidad": PROFILE_SEVERITY["P01"]},
    {"perfil": "P02", "nombre": PROFILE_LABELS["P02"],
     "descripcion": "Robo y hurto de bienes, incluidos robos a personas, hurto comercial, hurto patrimonial general y extorsión.",
     "submodalidades": "ROBO_PERSONAS; HURTO_COMERCIAL; HURTO_PATRIMONIAL; EXTORSION",
     "peso_severidad": PROFILE_SEVERITY["P02"]},
    {"perfil": "P03", "nombre": PROFILE_LABELS["P03"],
     "descripcion": "Burglary, allanamiento, intrusión y merodeo en inmuebles. No incluye robo de objetos dentro de vehículos, que se asigna a P01.",
     "submodalidades": "BURGLARY; INTRUSION",
     "peso_severidad": PROFILE_SEVERITY["P03"]},
    {"perfil": "P04", "nombre": PROFILE_LABELS["P04"],
     "descripcion": "Homicidios, agresiones físicas, lesiones, amenazas, violencia interpersonal, violencia doméstica explícita, violencia infantil, armas y delitos contra la libertad.",
     "submodalidades": "HOMICIDIO; VIOLENCIA_DOMESTICA; VIOLENCIA_INFANTIL; VIOLENCIA_INTERPERSONAL; ARMAS; ORDEN_PROTECCION; TRATA_PERSONAS",
     "peso_severidad": PROFILE_SEVERITY["P04"]},
    {"perfil": "P05", "nombre": PROFILE_LABELS["P05"],
     "descripcion": "Delitos sexuales, explotación sexual, exhibición/acoso sexual y incumplimiento de registro sexual.",
     "submodalidades": "AGRESION_SEXUAL; EXPLOTACION_SEXUAL; EXHIBICION_O_ACOSO; REGISTRO_SEXUAL",
     "peso_severidad": PROFILE_SEVERITY["P05"]},
    {"perfil": "P06", "nombre": PROFILE_LABELS["P06"],
     "descripcion": "Fraude, identidad, falsificación, malversación, vandalismo, daño a propiedad, incendio intencional y daño ambiental.",
     "submodalidades": "FRAUDE_IDENTIDAD; VANDALISMO_DANOS",
     "peso_severidad": PROFILE_SEVERITY["P06"]},
])

PROFILE_DEFINITIONS.to_csv("catalogo_perfiles.csv", index=False)

# Reglas ordenadas: lo específico va antes que lo general. La asignación es
# operativa y auditable; no pretende ser una taxonomía jurídica universal.
PROFILE_RULES = [
    ("P05", "AGRESION_SEXUAL", r"RAPE|SEXUAL PENETRATION|SODOMY|ORAL COPULATION|SEX,UNLAWFUL|BATTERY WITH SEXUAL CONTACT|BEASTIALITY|INCEST", "Alta", "Delito sexual explícito"),
    ("P05", "EXPLOTACION_SEXUAL", r"HUMAN TRAFFICKING - COMMERCIAL SEX|PIMPING|PANDERING", "Alta", "Explotación sexual"),
    ("P05", "EXHIBICION_O_ACOSO", r"INDECENT EXPOSURE|PEEPING|LEWD|LETTERS, LEWD|CHILD ANNOYING|CHILD PORNOGRAPHY", "Alta", "Conducta sexual/exhibición"),
    ("P05", "REGISTRO_SEXUAL", r"SEX OFFENDER REGISTRANT", "Media", "Incumplimiento de registro, no agresión"),
    ("P01", "VEHICULO_ROBADO", r"^VEHICLE - STOLEN$|^VEHICLE - ATTEMPT STOLEN$|^VEHICLE, STOLEN - OTHER|^BOAT - STOLEN$|DRIVING WITHOUT OWNER CONSENT", "Alta", "Sustracción de vehículo"),
    ("P01", "ACCESORIOS_VEHICULO", r"BURGLARY FROM VEHICLE|THEFT FROM MOTOR VEHICLE", "Alta", "Objetos/autopartes en vehículo"),
    ("P01", "BICICLETA_MOVILIDAD", r"^BIKE - STOLEN$|^BIKE - ATTEMPTED STOLEN$", "Alta", "Bicicleta/movilidad personal"),
    ("P01", "CONDUCCION", r"RECKLESS DRIVING|FAILURE TO YIELD|DRUNK ROLL", "Media", "Infracción de tránsito, no robo"),
    ("P03", "BURGLARY", r"^BURGLARY$|^BURGLARY, ATTEMPTED$", "Alta", "Ingreso ilícito a inmueble"),
    ("P03", "INTRUSION", r"^TRESPASSING$|^PROWLER$", "Media", "Intrusión/merodeo, no siempre robo"),
    ("P04", "HOMICIDIO", r"HOMICIDE|MANSLAUGHTER", "Alta", "Delito contra la vida"),
    ("P04", "VIOLENCIA_DOMESTICA", r"INTIMATE PARTNER", "Alta", "Violencia doméstica explícita"),
    ("P04", "VIOLENCIA_INFANTIL", r"CHILD ABUSE|CHILD NEGLECT|CRM AGNST CHLD|CHILD STEALING|CHILD ABANDONMENT|DRUGS, TO A MINOR", "Alta", "Violencia/abandono infantil"),
    ("P04", "ARMAS", r"WEAPONS POSSESSION|REPLICA FIREARMS|FIREARMS EPO|FIREARMS RO|BOMB SCARE", "Media", "Armas/amenaza, no violencia consumada"),
    ("P04", "VIOLENCIA_INTERPERSONAL", r"ASSAULT|BATTERY|LYNCHING|KIDNAPPING|FALSE IMPRISONMENT|STALKING|CRIMINAL THREATS|THREATENING|BRANDISH WEAPON|SHOTS FIRED|DISCHARGE FIREARMS|RESISTING ARREST|THROWING OBJECT AT MOVING VEHICLE", "Alta", "Violencia/amenaza contra personas"),
    ("P04", "ORDEN_PROTECCION", r"VIOLATION OF RESTRAINING ORDER|VIOLATION OF TEMPORARY RESTRAINING ORDER|VIOLATION OF COURT ORDER|CONTEMPT OF COURT", "Media", "Incumplimiento de orden, no implica violencia doméstica"),
    ("P04", "TRATA_PERSONAS", r"HUMAN TRAFFICKING - INVOLUNTARY SERVITUDE", "Alta", "Trata de personas no sexual"),
    ("P06", "FRAUDE_IDENTIDAD", r"THEFT OF IDENTITY|UNAUTHORIZED COMPUTER ACCESS|CREDIT CARDS|COUNTERFEIT|DOCUMENT FORGERY|DOCUMENT WORTHLESS|BUNCO|EMBEZZLEMENT|DEFRAUDING INNKEEPER|GRAND THEFT / INSURANCE FRAUD|BRIBERY|BIGAMY|FALSE POLICE REPORT", "Alta", "Fraude/engaño económico"),
    ("P06", "VANDALISMO_DANOS", r"VANDALISM|ARSON|ILLEGAL DUMPING|TELEPHONE PROPERTY - DAMAGE|CRUELTY TO ANIMALS", "Alta", "Daño a propiedad/entorno/animales"),
    ("P02", "ROBO_PERSONAS", r"^ROBBERY$|^ATTEMPTED ROBBERY$|THEFT, PERSON|PICKPOCKET|PURSE SNATCHING|THEFT FROM PERSON", "Alta", "Robo/hurto a personas"),
    ("P02", "HURTO_COMERCIAL", r"SHOPLIFTING|TILL TAP|DISHONEST EMPLOYEE", "Alta", "Hurto en comercio/empleado"),
    ("P02", "HURTO_PATRIMONIAL", r"THEFT PLAIN|THEFT-GRAND|THEFT, COIN MACHINE|PETTY THEFT - AUTO REPAIR|GRAND THEFT / AUTO REPAIR", "Alta", "Hurto patrimonial general"),
    ("P02", "EXTORSION", r"EXTORTION", "Media", "Extorsión patrimonial"),
]

# Ajustes de confianza y notas para casos que la regla captura pero cuyo
# encaje temático es discutible. La asignación se conserva para no perder
# volumen, pero se marca para revisión.
PROFILE_OVERRIDES = {
    "BRIBERY": ("P06", "FRAUDE_IDENTIDAD", "Media", "Corrupción/soborno; se agrupa con fraude económico"),
    "BIGAMY": ("P06", "FRAUDE_IDENTIDAD", "Media", "Delito legal/decepción; encaje temático débil"),
    "FALSE POLICE REPORT": ("P06", "FRAUDE_IDENTIDAD", "Media", "Engaño a autoridad; no es fraude económico típico"),
    "RECKLESS DRIVING": ("P01", "CONDUCCION", "Media", "Conducción temeraria; movilidad, no robo"),
    "FAILURE TO YIELD": ("P01", "CONDUCCION", "Media", "Infracción de tránsito; movilidad, no robo"),
    "DRUNK ROLL": ("P01", "CONDUCCION", "Baja", "Infracción de movilidad; revisar definición original"),
    "THREATENING PHONE CALLS/LETTERS": ("P04", "VIOLENCIA_INTERPERSONAL", "Alta", "Amenaza interpersonal"),
    "DRUGS, TO A MINOR": ("P04", "VIOLENCIA_INFANTIL", "Media", "Daño a menor; encaje temático parcial"),
    "OTHER MISCELLANEOUS CRIME": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Tipificación genérica; no permite asignación temática"),
    "DISTURBING THE PEACE": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Orden público; no encaja en los seis perfiles"),
    "CONTRIBUTING": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Contribución a delincuencia de menor; revisar"),
    "CONSPIRACY": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Delito genérico; requiere el delito base"),
    "DISRUPT SCHOOL": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Orden público escolar; no encaja"),
    "BLOCKING DOOR INDUCTION CENTER": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Orden público; no encaja"),
    "FAILURE TO DISPERSE": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Orden público; no encaja"),
    "INCITING A RIOT": ("NO_ASIGNADO", "SIN_PERFIL", "Baja", "Orden público; no encaja"),
}

def classify_profile(desc):
    s = str(desc).upper()
    if s in PROFILE_OVERRIDES:
        return PROFILE_OVERRIDES[s]
    for prof, sub, pattern, conf, note in PROFILE_RULES:
        if re.search(pattern, s):
            return prof, sub, conf, note
    return "NO_ASIGNADO", "SIN_PERFIL", "Baja", "Sin regla; requiere revisión"

rows = []
for desc, n in D["crime_description"].value_counts().items():
    prof, sub, conf, note = classify_profile(desc)
    rows.append((desc, n, prof, sub, conf, note))
PROFILE_AUDIT = pd.DataFrame(
    rows, columns=["crime_description", "incidentes", "perfil", "submodalidad", "confianza", "nota"]
).sort_values("incidentes", ascending=False)
PROFILE_AUDIT["porcentaje"] = 100 * PROFILE_AUDIT["incidentes"] / PROFILE_AUDIT["incidentes"].sum()

D = D.merge(
    PROFILE_AUDIT[["crime_description", "perfil", "submodalidad", "confianza", "nota"]],
    on="crime_description", how="left",
)
D["crime_profile"] = D["perfil"].fillna("NO_ASIGNADO")
D["crime_subprofile"] = D["submodalidad"].fillna("SIN_PERFIL")
D["profile_confidence"] = D["confianza"].fillna("Baja")
D["profile_note"] = D["nota"].fillna("Sin regla; requiere revisión")
D = D.drop(columns=["perfil", "submodalidad", "confianza", "nota"])

PROFILE_AUDIT.to_csv("auditoria_tipificaciones.csv", index=False)
print("Catálogo de perfiles:")
display(PROFILE_DEFINITIONS)
print("\nCobertura del mapeo:")
coverage = D["crime_profile"].ne("NO_ASIGNADO").mean()
print(f"  Asignado: {coverage:.2%} | NO_ASIGNADO: {1 - coverage:.2%}")
print(D.groupby("crime_profile").size().sort_values(ascending=False).to_string())
print("\nAuditoría (15 tipificaciones principales):")
display(PROFILE_AUDIT.head(15))
if (1 - coverage) > 0.02:
    print("ADVERTENCIA: más de 2 % sin asignar. Revisar las reglas y las notas de auditoría.")

# --- 5.4 Tipo de lugar ------------------------------------------------------
PREMISE_GROUPS = ["VIA_PUBLICA", "RESIDENCIA", "COMERCIO", "VEHICULO", "ESPACIO_PUBLICO", "OTROS"]
PREMISE_RULES = [
    ("VIA_PUBLICA", r"STREET|SIDEWALK|ALLEY|HIGHWAY|FREEWAY|PARKING|DRIVEWAY|BUS STOP|TRANSIT|ROAD|AVENUE|BLVD|BOULEVARD|ON STREET"),
    ("RESIDENCIA", r"RESIDENCE|HOUSE|APARTMENT|CONDO|GARAGE|YARD|PORCH|HOTEL|MOTEL|DWELLING|TRAILER|SINGLE FAMILY"),
    ("COMERCIO", r"STORE|SHOP|BUSINESS|BANK|RESTAURANT|BAR|OFFICE|MALL|MARKET|GAS STATION|RETAIL|SALON|LIQUOR|DEPARTMENT"),
    ("VEHICULO", r"VEHICLE|AUTO|CAR |TRUCK|BUS|MOTORCYCLE|BICYCLE|BIKE|MTA"),
    ("ESPACIO_PUBLICO", r"PARK|BEACH|PLAYGROUND|SCHOOL|CHURCH|HOSPITAL|LIBRARY|COLLEGE|UNIVERSITY|STADIUM|CEMETERY"),
]

def premise_group(desc):
    s = str(desc).upper()
    for label, pattern in PREMISE_RULES:
        if re.search(pattern, s):
            return label
    return "OTROS"

D["premise_group"] = D["premise_description"].map(premise_group)

# --- 5.5 Unidad territorial -------------------------------------------------
ZONE_COL = "reporting_district"
ZONE_FALLBACK = "area_name"
MIN_DISTRICT_INCIDENTS = 50

D["zone"] = (
    D[ZONE_COL].astype("string").str.strip()
    .replace({"NO_REGISTRADO": pd.NA, "": pd.NA, "0": pd.NA, "NAN": pd.NA})
    .fillna(D[ZONE_FALLBACK])
    .fillna("NO_REGISTRADO")
)

D = D.drop_duplicates().reset_index(drop=True)
D = D[D.zone.ne("NO_REGISTRADO") & D.month.notna()].copy()

# 2023-12 está incompleto: se excluye como mes de ocurrencia.
last_month = D["month"].max()
D = D[D["month"] < last_month].copy()

D["has_weapon"] = (~D["weapon_description"].isin(["NO_REGISTRADO", "NONE", "UNKNOWN"])).astype("int8")
D["has_modus"] = (~D["modus_operandi"].isin(["NO_REGISTRADO", "NONE", "UNKNOWN"])).astype("int8")

print(f"\nIncidentes utilizables: {len(D):,}")
print(f"Rango temporal: {D.month.min()} -> {D.month.max()}")
print(f"Zonas: {D.zone.nunique():,} | Tipificaciones: {D.crime_description.nunique():,}")
print("\nPerfiles consolidados:")
print(D.crime_profile.value_counts().to_string())
print("\nSubmodalidades principales:")
print(D.crime_subprofile.value_counts().head(15).to_string())
print("\nTipos de lugar:")
print(D.premise_group.value_counts().to_string())
print(f"\nRetrasos negativos reales: {int(D.report_delay_negative.sum()):,}")
print(f"Reportes el mismo día: {int(D.same_day_report.sum()):,}")
print(f"Coordenadas válidas: {int(D.valid_geo.sum()):,} / {len(D):,}")

del D["date_reported"]
_ = gc.collect()

## 6. Panel temporal: zona × perfil × mes

Se construye una grilla completa de zonas, perfiles y meses. Una combinación sin incidentes en un mes cubierto se registra como **0**; un mes sin cobertura de la zona se registra como faltante y se marca con `covered=False`. Después se crean rezagos, medias móviles y la variable objetivo (incidentes del mes siguiente).

**Advertencia de cobertura:** la presencia de incidentes no demuestra cobertura completa de la zona; los ceros se interpretan como ausencia de registros *en el archivo*, no ausencia real de delitos. Las zonas sin actividad observada se excluyen en lugar de imputarse a cero.

In [ ]:
monthly = (
    D[D.crime_profile.isin(PROFILE_CODES)]
    .groupby(["zone", "crime_profile", "month"], observed=True)
    .size().rename("incidents").reset_index()
)
zone_month = (
    D.groupby(["zone", "month"], observed=True)
    .size().rename("zone_total").reset_index()
)
zone_month["covered"] = True

zones = sorted(D["zone"].unique())
months = pd.period_range(D["month"].min(), D["month"].max(), freq="M")
print(f"Zonas: {len(zones):,} | Meses: {len(months)} | Celdas base: {len(zones) * len(months):,}")

grid = pd.MultiIndex.from_product([zones, months], names=["zone", "month"]).to_frame(index=False)
grid = grid.merge(zone_month, on=["zone", "month"], how="left")
grid["covered"] = grid["covered"].fillna(False).astype(bool)
grid["zone_total"] = grid["zone_total"].fillna(0).astype("float32")

prof_df = pd.DataFrame({"crime_profile": PROFILE_CODES})
panel = grid.merge(prof_df, how="cross")
panel = panel.merge(monthly, on=["zone", "crime_profile", "month"], how="left")
panel.loc[panel["covered"], "incidents"] = panel.loc[panel["covered"], "incidents"].fillna(0)
panel = panel.sort_values(["zone", "crime_profile", "month"]).reset_index(drop=True)

# Rezagos por serie distrito-perfil.
g = panel.groupby(["zone", "crime_profile"], sort=False)["incidents"]
for lag in [0, 1, 2, 3, 6, 12]:
    panel[f"lag_{lag}"] = g.shift(lag)
panel["target_next_month"] = g.shift(-1)

# Medias móviles y volatilidad reciente.
for window in [3, 6]:
    panel[f"rolling_mean_{window}"] = (
        g.transform(lambda s: s.rolling(window, min_periods=window).mean())
    )
panel["rolling_std_3"] = (
    g.transform(lambda s: s.rolling(3, min_periods=3).std())
)

# Estacionalidad y tendencia.
month_index = {m: i for i, m in enumerate(months)}
panel["month_num"] = panel["month"].dt.month
panel["year"] = panel["month"].dt.year
panel["month_sin"] = np.sin(2 * np.pi * panel["month_num"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["month_num"] / 12)
panel["trend"] = panel["month"].map(month_index).astype("float32")
panel["zone_total_log"] = np.log1p(panel["zone_total"])
panel["target_month"] = panel["month"] + 1

target_months = months[1:]
panel = panel[panel["covered"] & panel["target_month"].isin(target_months)].copy()

LAG_COLS = ["lag_0", "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
            "rolling_mean_3", "rolling_mean_6", "rolling_std_3"]
model_data = panel.dropna(subset=LAG_COLS + ["target_next_month"]).copy()

coverage = panel.groupby("month")["covered"].mean()
print(f"Cobertura media de la grilla: {panel['covered'].mean():.2%}")
print(f"Celdas con historial suficiente: {len(model_data):,} de {len(panel):,}")
print(f"Targets disponibles: {model_data.target_month.min()} -> {model_data.target_month.max()}")
print("\nCobertura por mes (primeros/últimos):")
print(pd.concat([coverage.head(3), coverage.tail(3)]).round(3).to_string())

del grid, monthly, zone_month, prof_df, coverage
_ = gc.collect()

## 7. División temporal honesta

La división se hace por **mes objetivo**, no por filas aleatorias. El conjunto de entrenamiento contiene meses anteriores al de validación, y validación anterior al de prueba. Así se evita que el modelo vea el futuro de la misma serie. La fecha de corte del entrenamiento se usa después para calcular los perfiles y las variables territoriales estáticas.

In [ ]:
# Calibración histórica fija: 12 meses iniciales completos, ANTES del primer
# mes objetivo que entrará en cualquiera de los modelos supervisados.
PROFILE_CALIBRATION_MONTHS = 12
available_months = sorted(D["month"].dropna().unique())
if len(available_months) < PROFILE_CALIBRATION_MONTHS + 18:
    raise ValueError("Se requieren al menos 30 meses para calibración y evaluación temporal.")
PROFILE_CUTOFF = available_months[PROFILE_CALIBRATION_MONTHS - 1]
unique_targets = sorted(m for m in model_data["target_month"].dropna().unique()
                        if m > PROFILE_CUTOFF + 1)
print(f"Perfiles fijos hasta {PROFILE_CUTOFF}; objetivos posteriores: {len(unique_targets)} meses")
n_targets = len(unique_targets)
n_test = max(2, int(round(0.15 * n_targets)))
n_val = max(2, int(round(0.15 * n_targets)))
n_train = n_targets - n_test - n_val
if n_train < 6:
    raise ValueError(
        f"Historial insuficiente para dividir: {n_targets} meses objetivo. "
        "Usa el dataset completo o más filas."
    )

train_targets = unique_targets[:n_train]
val_targets = unique_targets[n_train:n_train + n_val]
test_targets = unique_targets[n_train + n_val:]

train = model_data[model_data["target_month"].isin(train_targets)].copy()
val = model_data[model_data["target_month"].isin(val_targets)].copy()
test = model_data[model_data["target_month"].isin(test_targets)].copy()

TRAIN_TARGET_MAX = max(train_targets)

assert train["target_month"].max() < val["target_month"].min()
assert val["target_month"].max() < test["target_month"].min()
assert set(train["target_month"]).isdisjoint(set(test["target_month"]))

print(f"Meses objetivo: train={len(train_targets)} | val={len(val_targets)} | test={len(test_targets)}")
print(f"Train: {train.target_month.min()} -> {train.target_month.max()} ({len(train):,} filas)")
print(f"Val:   {val.target_month.min()} -> {val.target_month.max()} ({len(val):,} filas)")
print(f"Test:  {test.target_month.min()} -> {test.target_month.max()} ({len(test):,} filas)")
print(f"Corte de calibración de perfiles: {PROFILE_CUTOFF}")
assert PROFILE_CUTOFF < min(train_targets) - 1

## 8. Perfilamiento temático por zona

Para cada zona se calcula la **mezcla de los seis perfiles** durante el periodo de calibración. Esa mezcla permite:

- identificar el **perfil principal** y hasta tres **perfiles secundarios** con umbrales mínimos de evidencia;
- marcar zonas **mixtas** cuando ningún perfil domina con claridad;
- calcular una **prioridad** multi-criterio separada de la frecuencia;
- describir el escenario dominante (tipo de lugar, horario, fin de semana, arma y modus operandi).

Las variables se calculan **solo con el periodo de entrenamiento** para no filtrar información del futuro. Los perfiles se fijan con los primeros 12 meses, y las filas supervisadas empiezan después.

**Esta unidad no segmenta territorios por similitud estadística.** El objetivo no es descubrir cuántos tipos de zonas existen, sino medir qué perfiles predominan en cada territorio, cómo se combinan y cuáles deben priorizarse.

In [ ]:
# Periodo de calibración ESTRICTAMENTE anterior a todas las observaciones
# supervisadas. Ningún perfil utiliza incidentes de meses objetivo posteriores.
profile_data = D[D["month"] <= PROFILE_CUTOFF].copy()
assert profile_data["month"].max() < min(train_targets) - 1
n_train_months = profile_data["month"].nunique()
print(f"Periodo de perfilamiento: {profile_data.month.min()} -> {profile_data.month.max()} ({n_train_months} meses)")

zone_counts = profile_data.groupby("zone").size()
valid_zones = zone_counts[zone_counts >= MIN_DISTRICT_INCIDENTS].index
profile_data = profile_data[profile_data["zone"].isin(valid_zones)].copy()
print(f"Zonas para perfiles: {len(valid_zones):,} (>= {MIN_DISTRICT_INCIDENTS} incidentes)")

# --- 8.1 Matriz zona × perfil ------------------------------------------------
zone_profile_counts = pd.crosstab(profile_data["zone"], profile_data["crime_profile"])
zone_profile_counts = (
    zone_profile_counts.reindex(index=sorted(valid_zones), columns=PROFILE_CODES, fill_value=0)
    .astype("int64")
)
zone_sin = (
    profile_data[profile_data.crime_profile.eq("NO_ASIGNADO")]
    .groupby("zone").size().reindex(zone_profile_counts.index, fill_value=0)
)
zone_total_all = zone_profile_counts.sum(axis=1) + zone_sin
zone_profile_shares = zone_profile_counts.div(zone_total_all.replace(0, np.nan), axis=0).fillna(0)
sin_asignar_share = (zone_sin / zone_total_all.replace(0, np.nan)).fillna(0)

matriz_zona_perfil = zone_profile_counts.copy()
for p in PROFILE_CODES:
    matriz_zona_perfil[f"share_{p}"] = zone_profile_shares[p]
matriz_zona_perfil["sin_asignar"] = zone_sin
matriz_zona_perfil["sin_asignar_share"] = sin_asignar_share
matriz_zona_perfil = matriz_zona_perfil.reset_index()

# --- 8.2 Escenario dominante por zona × perfil ------------------------------
situ = (
    profile_data[profile_data.crime_profile.isin(PROFILE_CODES)]
    .groupby(["zone", "crime_profile"])
    .agg(
        incidentes=("zone", "size"),
        night_share=("night", "mean"),
        weekend_share=("weekend", "mean"),
        weapon_share=("has_weapon", "mean"),
        modus_share=("has_modus", "mean"),
    ).reset_index()
)

prem_counts = (
    profile_data[profile_data.crime_profile.isin(PROFILE_CODES)]
    .groupby(["zone", "crime_profile", "premise_group"]).size().rename("n").reset_index()
)
prem_counts["share_premise"] = prem_counts["n"] / prem_counts.groupby(["zone", "crime_profile"])["n"].transform("sum")
prem_top = (
    prem_counts.sort_values(["zone", "crime_profile", "n"], ascending=[True, True, False])
    .drop_duplicates(["zone", "crime_profile"])
    .rename(columns={"premise_group": "premise_top", "share_premise": "premise_top_share"})
)

shares_long = zone_profile_shares.stack().rename("share_zona").reset_index()
shares_long.columns = ["zone", "crime_profile", "share_zona"]
zone_profile_long = (
    situ.merge(prem_top[["zone", "crime_profile", "premise_top", "premise_top_share"]],
               on=["zone", "crime_profile"], how="left")
    .merge(shares_long, on=["zone", "crime_profile"], how="left")
)

global_counts = profile_data[profile_data.crime_profile.isin(PROFILE_CODES)].crime_profile.value_counts()
global_shares = (global_counts / global_counts.sum()).rename("share_global")
zone_profile_long["share_global"] = zone_profile_long["crime_profile"].map(global_shares).fillna(0)
zone_profile_long["lift"] = zone_profile_long["share_zona"] / zone_profile_long["share_global"].replace(0, np.nan)

# Tendencia reciente dentro de la ventana de calibración: últimos 3 meses vs. 3 previos.
monthly_profile = (
    profile_data[profile_data.crime_profile.isin(PROFILE_CODES)]
    .groupby(["zone", "crime_profile", "month"]).size().rename("n").reset_index()
)
cal_months = sorted(profile_data["month"].unique())
month_pos = {m: i for i, m in enumerate(cal_months)}
monthly_profile["t"] = monthly_profile["month"].map(month_pos)
last_t = monthly_profile["t"].max()
recent = monthly_profile[monthly_profile.t > last_t - 3].groupby(["zone", "crime_profile"])["n"].mean()
prev = monthly_profile[(monthly_profile.t <= last_t - 3) & (monthly_profile.t > last_t - 6)].groupby(["zone", "crime_profile"])["n"].mean()
trend_ratio = ((recent + 1) / (prev + 1)).rename("trend_ratio")
zone_profile_long = zone_profile_long.merge(trend_ratio, on=["zone", "crime_profile"], how="left")
zone_profile_long["trend_ratio"] = zone_profile_long["trend_ratio"].fillna(1.0)

# --- 8.3 Prioridad multi-criterio (no es frecuencia) -------------------------
zone_profile_long["severity"] = zone_profile_long["crime_profile"].map(PROFILE_SEVERITY)
zone_profile_long["share_norm"] = zone_profile_long["share_zona"].clip(0, 1)
zone_profile_long["lift_norm"] = (zone_profile_long["lift"] / 2).clip(0, 1)
zone_profile_long["trend_norm"] = ((zone_profile_long["trend_ratio"] - 0.5) / 1.5).clip(0, 1)
zone_profile_long["evidence_norm"] = (
    np.log1p(zone_profile_long["incidentes"]) / np.log1p(100)
).clip(0, 1)
zone_profile_long["priority_score"] = (
    0.40 * zone_profile_long["share_norm"]
    + 0.25 * zone_profile_long["lift_norm"]
    + 0.15 * zone_profile_long["trend_norm"]
    + 0.10 * zone_profile_long["severity"]
    + 0.10 * zone_profile_long["evidence_norm"]
)
zone_profile_long["priority_rank"] = (
    zone_profile_long.groupby("zone")["priority_score"]
    .rank(ascending=False, method="first").astype(int)
)

# --- 8.4 Perfil principal, secundarios y evidencia --------------------------
SECONDARY_MIN_SHARE = 0.10
SECONDARY_MIN_INCIDENTS = 10

summary_rows = []
for zone, g in zone_profile_long.groupby("zone", sort=False):
    g = g.sort_values("share_zona", ascending=False).reset_index(drop=True)
    main = g.iloc[0]
    total = float(g["incidentes"].sum())
    sec = g.iloc[1:]
    sec = sec[(sec["share_zona"] >= SECONDARY_MIN_SHARE) & (sec["incidentes"] >= SECONDARY_MIN_INCIDENTS)].head(3)
    top1 = float(main["share_zona"])
    top2 = float(g["share_zona"].iloc[1]) if len(g) > 1 else 0.0
    evidence = "Alta" if main["incidentes"] >= 50 else ("Media" if main["incidentes"] >= 10 else "Baja")
    top_priority = g.sort_values("priority_score", ascending=False).iloc[0]
    row = {
        "zone": zone,
        "incidentes_totales": total,
        "perfil_principal": main["crime_profile"],
        "perfil_principal_nombre": PROFILE_LABELS.get(main["crime_profile"], main["crime_profile"]),
        "perfil_principal_share": top1,
        "perfil_principal_incidentes": int(main["incidentes"]),
        "perfil_principal_evidencia": evidence,
        "n_perfiles_relevantes": 1 + len(sec),
        "zona_mixta": bool((top1 < 0.30) or ((top1 - top2) < 0.05)),
        "perfil_secundario_1": sec.iloc[0]["crime_profile"] if len(sec) >= 1 else "NINGUNO",
        "perfil_secundario_1_share": float(sec.iloc[0]["share_zona"]) if len(sec) >= 1 else 0.0,
        "perfil_secundario_2": sec.iloc[1]["crime_profile"] if len(sec) >= 2 else "NINGUNO",
        "perfil_secundario_2_share": float(sec.iloc[1]["share_zona"]) if len(sec) >= 2 else 0.0,
        "perfil_secundario_3": sec.iloc[2]["crime_profile"] if len(sec) >= 3 else "NINGUNO",
        "perfil_secundario_3_share": float(sec.iloc[2]["share_zona"]) if len(sec) >= 3 else 0.0,
        "perfil_prioritario": top_priority["crime_profile"],
        "prioridad_score": float(top_priority["priority_score"]),
        "prioridad_es_principal": bool(top_priority["crime_profile"] == main["crime_profile"]),
        "sin_asignar_share": float(sin_asignar_share.get(zone, 0.0)),
    }
    summary_rows.append(row)

zone_profile_summary = pd.DataFrame(summary_rows)
zone_profile_summary["prioridad_flag"] = pd.cut(
    zone_profile_summary["prioridad_score"],
    bins=[-0.001, 0.40, 0.60, 1.001],
    labels=["Baja", "Media", "Alta"],
).astype(str)

print(f"Zonas con perfil principal: {zone_profile_summary['perfil_principal'].ne('SIN_EVIDENCIA').sum():,}")
print("Distribución del perfil principal:")
print(zone_profile_summary["perfil_principal"].value_counts().to_string())
print("\nZonas mixtas:", int(zone_profile_summary["zona_mixta"].sum()))
print("Perfil prioritario distinto del principal:", int((~zone_profile_summary["prioridad_es_principal"]).sum()))
display(zone_profile_summary.head(10))

# --- 8.5 Unión al panel de modelado -----------------------------------------
zone_mean_total = profile_data.groupby("zone").size() / n_train_months
zone_mean_total_log = np.log1p(zone_mean_total).rename("zone_mean_total_log")

zone_profile_cal = (
    profile_data[profile_data.crime_profile.isin(PROFILE_CODES)]
    .groupby(["zone", "crime_profile"]).size().rename("profile_inc").reset_index()
)
zone_total = (
    profile_data[profile_data.crime_profile.isin(PROFILE_CODES)]
    .groupby("zone").size().rename("total").reset_index()
)
zone_profile_cal = zone_profile_cal.merge(zone_total, on="zone", how="left")
zone_profile_cal["zone_profile_share"] = zone_profile_cal["profile_inc"] / zone_profile_cal["total"]

model_data = model_data.merge(
    zone_profile_cal[["zone", "crime_profile", "zone_profile_share"]],
    on=["zone", "crime_profile"], how="left",
)
model_data = model_data.merge(zone_mean_total_log, on="zone", how="left")
model_data[["zone_profile_share", "zone_mean_total_log"]] = (
    model_data[["zone_profile_share", "zone_mean_total_log"]].fillna(0)
)

print(f"Filas tras unir perfiles: {len(model_data):,}")
print(f"Zonas modeladas: {model_data.zone.nunique():,} | Perfiles: {model_data.crime_profile.nunique()}")

# Recrear las particiones con las nuevas columnas.
train = model_data[model_data["target_month"].isin(train_targets)].copy()
val = model_data[model_data["target_month"].isin(val_targets)].copy()
test = model_data[model_data["target_month"].isin(test_targets)].copy()
assert train["target_month"].max() < val["target_month"].min() < test["target_month"].min()
assert PROFILE_CUTOFF < min(train_targets) - 1

del profile_data
_ = gc.collect()

## 9. Matriz supervisada

Las variables numéricas se estandarizan con la media y desviación del **entrenamiento**. El perfil delictivo se codifica con one-hot. El target es `target_next_month` (conteo de incidentes del perfil en el mes siguiente). No se usan variables del mes objetivo ni etiquetas de agrupamiento territorial.

La matriz incluye historial (`lag_0`, lags 1/2/3/6/12), medias móviles, volatilidad, calendario, tendencia, volumen de la zona y participación histórica del perfil en la zona.

In [ ]:
NUM_FEATURES = [
    "lag_0", "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "rolling_mean_3", "rolling_mean_6", "rolling_std_3",
    "month_sin", "month_cos", "trend",
    "zone_total_log", "zone_mean_total_log", "zone_profile_share",
]
CAT_FEATURE = "crime_profile"

model_data = model_data.dropna(subset=NUM_FEATURES + ["target_next_month"]).copy()
train = model_data[model_data["target_month"].isin(train_targets)].copy()
val = model_data[model_data["target_month"].isin(val_targets)].copy()
test = model_data[model_data["target_month"].isin(test_targets)].copy()

num_scaler = StandardScaler()
X_train_num = num_scaler.fit_transform(train[NUM_FEATURES]).astype("float32")
X_val_num = num_scaler.transform(val[NUM_FEATURES]).astype("float32")
X_test_num = num_scaler.transform(test[NUM_FEATURES]).astype("float32")

cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
X_train_cat = cat_encoder.fit_transform(train[[CAT_FEATURE]])
X_val_cat = cat_encoder.transform(val[[CAT_FEATURE]])
X_test_cat = cat_encoder.transform(test[[CAT_FEATURE]])

X_train = np.hstack([X_train_num, X_train_cat]).astype("float32")
X_val = np.hstack([X_val_num, X_val_cat]).astype("float32")
X_test = np.hstack([X_test_num, X_test_cat]).astype("float32")

y_train = train["target_next_month"].to_numpy(dtype="float32")
y_val = val["target_next_month"].to_numpy(dtype="float32")
y_test = test["target_next_month"].to_numpy(dtype="float32")

# Cota conservadora: evita que expm1 extrapole a conteos nunca observados.
LOG_CAP = float(np.log1p(y_train.max()))
print(f"Cota log1p para predicciones: {LOG_CAP:.3f} (máximo train={y_train.max():.0f})")

for name, arr in [("train", X_train), ("val", X_val), ("test", X_test)]:
    assert np.isfinite(arr).all(), f"{name} contiene NaN/inf"
    print(f"{name}: {arr.shape}")
print(f"Perfiles one-hot: {list(cat_encoder.get_feature_names_out([CAT_FEATURE]))}")
print(f"Target: media={y_train.mean():.2f} | max={y_train.max():.0f}")

## 10. Arquitectura de la MLP

Una sola red, con la misma arquitectura y el mismo presupuesto de entrenamiento que las referencias:

- Entrada: historial, estacionalidad, volumen, participación histórica del perfil y one-hot del perfil.
- Tres bloques densos con BatchNormalization, activación ReLU y Dropout.
- Salida `softplus` para garantizar conteos no negativos.

Como los conteos tienen cola larga y muchos valores pequeños, el entrenamiento se hace sobre `log1p(target)` y las predicciones se devuelven con `expm1`; las métricas finales se calculan en la escala original de incidentes. La pérdida es Huber sobre la escala logarítmica, robusta a conteos extremos.

**Interpretación:** la red aprende el patrón temporal de cada zona-perfil. No se agrega una etiqueta de agrupamiento territorial porque el objetivo es describir perfiles y pronosticar.

In [ ]:
def build_mlp(input_dim, name):
    tf.keras.utils.set_random_seed(SEED)
    inputs = Input(shape=(input_dim,), name="features")
    x = Dense(128, activation="relu", name="dense_1")(inputs)
    x = BatchNormalization(name="bn_1")(x)
    x = Dropout(0.10, name="drop_1")(x)
    x = Dense(64, activation="relu", name="dense_2")(x)
    x = BatchNormalization(name="bn_2")(x)
    x = Dropout(0.10, name="drop_2")(x)
    x = Dense(32, activation="relu", name="dense_3")(x)
    x = BatchNormalization(name="bn_3")(x)
    outputs = Dense(1, activation="softplus", name="incidents_next_month")(x)
    model = Model(inputs, outputs, name=name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.Huber(delta=0.5),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


model = build_mlp(X_train.shape[1], "MLP_incidentes")
model.summary()

## 11. Entrenamiento

Se usa `EarlyStopping` sobre el MAE de validación con restauración de los mejores pesos, `ReduceLROnPlateau` para ajustar la tasa de aprendizaje y `ModelCheckpoint` para guardar el mejor modelo en `mlp_perfilamiento.keras`. El número máximo de épocas es una cota; el modelo decide cuándo detenerse. No se usa el conjunto de prueba durante el entrenamiento.

In [ ]:
def make_callbacks():
    return [
        EarlyStopping(monitor="val_mae", mode="min", patience=10, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_mae", mode="min", factor=0.5, patience=5, min_lr=1e-5, verbose=0),
        ModelCheckpoint("mlp_perfilamiento.keras", monitor="val_mae", mode="min",
                        save_best_only=True, verbose=0),
    ]

BATCH_SIZE = 512
MAX_EPOCHS = 100

# El target se transforma solo para el entrenamiento; la evaluación vuelve a conteos.
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

history = model.fit(
    X_train, y_train_log,
    validation_data=(X_val, y_val_log),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=make_callbacks(), verbose=0,
)

print(f"MLP: {len(history.history['loss'])} épocas | "
      f"val_loss={min(history.history['val_loss']):.3f} | "
      f"val_mae={min(history.history['val_mae']):.3f}")

## 12. Evaluación y comparación

Las predicciones se recortan en cero y se limitan al máximo de conteos observado en entrenamiento para evitar que la transformación exponencial extrapole a valores absurdos. Se reportan MAE, RMSE y R² en el conjunto de prueba. La persistencia (`lag_0`) es la referencia mínima: si un modelo no supera la persistencia, no aporta valor predictivo. Los árboles y la MLP usan exactamente las mismas particiones y matrices.

No se comparan variantes de agrupamiento territorial: esa discusión pertenece a la Unidad 2 y no forma parte del producto final. Acá se compara la MLP con referencias supervisadas y con la persistencia.

In [ ]:
def metricas(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float).ravel(), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


preds = {
    "Persistencia": test["lag_0"].to_numpy(dtype=float),
    "MLP": np.expm1(np.clip(model.predict(X_test, verbose=0).ravel(), 0, LOG_CAP)),
}

# Referencias de árboles con la misma matriz supervisada.
hgb = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
    early_stopping=False, random_state=SEED,
)
hgb.fit(X_train, y_train)
preds["HGB"] = hgb.predict(X_test)

rf = RandomForestRegressor(
    n_estimators=120, min_samples_leaf=5, n_jobs=-1, random_state=SEED,
)
rf.fit(X_train, y_train)
preds["RF"] = rf.predict(X_test)

comparison = pd.DataFrame(
    [{"Modelo": name, **metricas(y_test, p)} for name, p in preds.items()]
).sort_values("MAE").reset_index(drop=True)

display(comparison.round(4))
best_model = comparison.iloc[0]["Modelo"]
print(f"Mejor MAE: {best_model} ({comparison.iloc[0]['MAE']:.4f})")
print(f"Persistencia MAE: {comparison.loc[comparison.Modelo == 'Persistencia', 'MAE'].iloc[0]:.4f}")

## 13. Curvas de aprendizaje y diagnóstico visual

Se revisa si el entrenamiento se estabilizó, si hay sobreajuste y cómo se distribuyen los errores. Los gráficos se generan a partir de la historia real de esta ejecución.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("MLP — pérdida Huber (log1p)")
axes[0].set_xlabel("Época")
axes[0].legend()

axes[1].plot(history.history["mae"], label="train")
axes[1].plot(history.history["val_mae"], label="val")
axes[1].set_title("MLP — MAE (log1p)")
axes[1].set_xlabel("Época")
axes[1].legend()
plt.tight_layout()
plt.show()

pred_mlp = np.clip(preds["MLP"], 0, None)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(y_test, pred_mlp, alpha=0.25, s=12)
lim = [0, max(y_test.max(), pred_mlp.max()) * 1.05]
axes[0].plot(lim, lim, "r--", linewidth=1)
axes[0].set_xlabel("Incidentes observados")
axes[0].set_ylabel("Incidentes predichos")
axes[0].set_title("MLP: observado vs predicho")

residuals = y_test - pred_mlp
axes[1].hist(residuals, bins=60, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residuo (observado - predicho)")
axes[1].set_title("Distribución de residuos")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    sns.barplot(data=comparison, x=metric, y="Modelo", ax=ax, palette="viridis")
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 14. Análisis de errores y prioridad

No hay un “aporte del agrupamiento territorial” que medir: esa comparación se eliminó. Lo que sí se evalúa es el error del modelo por perfil delictivo, por nivel de volumen de la zona, los mayores errores y los incrementos previstos. También se cruza el pronóstico con el perfil principal y la prioridad de la zona para apoyar la lectura de negocio.

**No confundir:** el perfil delictivo es el target temático; la prioridad es una decisión multi-criterio; el pronóstico es una estimación de volumen. Son tres productos distintos.

In [ ]:
test_eval = test.copy()
for name, p in preds.items():
    test_eval[name] = np.clip(np.asarray(p, dtype=float).ravel(), 0, None)

# Error por perfil delictivo.
err_profile = {}
for name in preds:
    tmp = test_eval[["crime_profile"]].copy()
    tmp["ae"] = (test_eval["target_next_month"] - test_eval[name]).abs()
    err_profile[name] = tmp.groupby("crime_profile")["ae"].mean()
err_profile = pd.DataFrame(err_profile)
print("MAE por perfil delictivo:")
display(err_profile.round(3))

# Error por decil de volumen de la zona.
test_eval["vol_decile"] = pd.qcut(
    test_eval["zone_mean_total_log"], 10, labels=False, duplicates="drop"
)
err_vol = {}
for name in preds:
    tmp = test_eval[["vol_decile"]].copy()
    tmp["ae"] = (test_eval["target_next_month"] - test_eval[name]).abs()
    err_vol[name] = tmp.groupby("vol_decile")["ae"].mean()
err_vol = pd.DataFrame(err_vol)
print("MAE por decil de volumen (0 = zonas de menor volumen):")
display(err_vol.round(3))

# Mayores errores.
test_eval["error_abs"] = (test_eval["target_next_month"] - test_eval["MLP"]).abs()
top_errors = (
    test_eval.sort_values("error_abs", ascending=False)
    [["zone", "crime_profile", "target_month", "target_next_month", "lag_0", "MLP", "error_abs"]]
    .head(20)
)
print("\nMayores errores absolutos (MLP):")
display(top_errors.round(2))

# Incrementos previstos: predicción menos el último valor observado.
test_eval["delta_pred"] = test_eval["MLP"] - test_eval["lag_0"]
growth = (
    test_eval.groupby("crime_profile")
    .agg(
        delta_medio=("delta_pred", "mean"),
        delta_mediano=("delta_pred", "median"),
        observado_medio=("target_next_month", "mean"),
        n=("delta_pred", "size"),
    )
    .reset_index()
    .sort_values("delta_medio", ascending=False)
)
print("\nIncrementos previstos por perfil:")
display(growth.round(2))

# Lectura por perfil principal de la zona.
zone_priority = zone_profile_summary[["zone", "perfil_principal", "perfil_prioritario", "prioridad_score"]]
growth_zone = (
    test_eval.merge(zone_priority, on="zone", how="left")
    .groupby(["perfil_principal", "crime_profile"])
    .agg(
        delta_medio=("delta_pred", "mean"),
        observado_medio=("target_next_month", "mean"),
        n=("delta_pred", "size"),
    )
    .reset_index()
    .sort_values("delta_medio", ascending=False)
)
print("\nIncrementos previstos según perfil principal de la zona:")
display(growth_zone.head(15).round(2))

## 15. Diagnóstico de perfiles temáticos

Los perfiles están congelados al final de la calibración histórica. Se muestra la composición de las zonas con más incidentes y la distribución del perfil principal. **No se interpreta una concentración de denuncias como tasa de criminalidad.** Los resultados son retrospectivos y las predicciones de prueba son simulaciones históricas, no pronósticos actuales.

La tabla de zonas debe leerse en tres capas: (1) perfil principal y secundarios, (2) evidencia y mezcla, (3) prioridad. Una zona puede tener un perfil principal patrimonial y, aun así, una prioridad alta en un perfil violento poco frecuente pero sobrerrepresentado.

In [ ]:
share_cols = [f"share_{p}" for p in PROFILE_CODES]
top_zones = zone_profile_summary.sort_values("incidentes_totales", ascending=False).head(20)["zone"]
heat = matriz_zona_perfil.set_index("zone").loc[top_zones, share_cols]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
axes[0].set_title("Composición de perfiles: 20 zonas con más incidentes")
axes[0].set_xlabel("Perfil")
axes[0].set_ylabel("Zona")

main_counts = zone_profile_summary["perfil_principal"].value_counts().reindex(PROFILE_CODES, fill_value=0)
main_counts.plot(kind="bar", color="steelblue", ax=axes[1])
axes[1].set_title("Zonas por perfil principal")
axes[1].set_xlabel("Perfil")
axes[1].set_ylabel("Zonas")
plt.tight_layout()
plt.show()

# Inspección del riesgo de fuga temporal.
print("Calibración:", D.month.min(), "a", PROFILE_CUTOFF)
print("Primer mes objetivo de entrenamiento:", min(train_targets))
assert PROFILE_CUTOFF < min(train_targets) - 1
print("OK: perfiles construidos antes del inicio de las predicciones supervisadas")

print("\nMAE por perfil delictivo:")
display(err_profile.round(3))

## 16. Lectura de resultados, apoyo a decisiones y límites

La tabla de comparación es la evidencia. No se fuerza que la red gane: si la MLP queda cerca de HGB y supera la persistencia, es competitiva. Si la persistencia es mejor en MAE, el modelo no aporta valor predictivo y hay que decirlo.

**Sobre los perfiles.** El perfil principal es una descripción de la mezcla de la zona, no una etiqueta exclusiva. Los perfiles secundarios importan: una zona puede ser principalmente patrimonial y tener un problema relevante de violencia o de delitos sexuales. La prioridad multi-criterio es una herramienta de discusión, no una decisión automática; sus pesos de severidad son explícitos y discutibles.

**Uso responsable.** Los resultados pueden apoyar la asignación de recursos, el diseño de campañas de prevención y la priorización de análisis. No deben usarse para estigmatizar barrios, justificar vigilancia masiva ni inferir culpabilidad. Los conteos dependen de la denuncia; una caída puede reflejar subregistro.

**Límites.** El modelo no incorpora población, movilidad, clima, eventos ni política de policía. Los perfiles agrupan tipificaciones heterogéneas y algunas asignaciones son discutibles; la auditoría permite revisarlas. Los perfiles son una fotografía del periodo de entrenamiento y pueden cambiar. La serie es corta para aprendizaje profundo; una MLP tabular es adecuada, pero no garantiza superioridad. La validación es temporal, no aleatoria, y el test es un único periodo.

In [ ]:
# Tabla final de zonas (descriptiva): perfil principal, secundarios, evidencia y prioridad.
final_zones = zone_profile_summary.sort_values("incidentes_totales", ascending=False).copy()
display(final_zones.head(25).round(3))

# Resumen textual basado en los resultados calculados.
print("=" * 82)
print("RESUMEN DE LA EJECUCIÓN")
print("=" * 82)
print(f"Incidentes analizados: {len(D):,}")
print(f"Zonas: {D.zone.nunique():,} | Tipificaciones: {D.crime_description.nunique():,}")
print(f"Perfiles consolidados: {len(PROFILE_CODES)} | Cobertura: {D.crime_profile.ne('NO_ASIGNADO').mean():.2%}")
print(f"Zonas con perfil principal: {zone_profile_summary['perfil_principal'].ne('SIN_EVIDENCIA').sum():,}")
print(f"Zonas mixtas: {int(zone_profile_summary['zona_mixta'].sum()):,}")
print(f"Zonas con prioridad distinta al perfil principal: {int((~zone_profile_summary['prioridad_es_principal']).sum()):,}")
print(f"Filas de modelado: {len(model_data):,}")
print(f"Test: {test.target_month.min()} -> {test.target_month.max()} ({len(test):,} filas)")
print("-" * 82)
print(comparison.round(4).to_string(index=False))
print("-" * 82)
print(f"Mejor modelo por MAE: {best_model}")
print("Modelos supervisados comparados.")
print("=" * 82)

# Exportación de resultados para el informe, la API y la aplicación web.
comparison.to_csv("resultados_modelos.csv", index=False)
PROFILE_AUDIT.to_csv("auditoria_tipificaciones.csv", index=False)
PROFILE_DEFINITIONS.to_csv("catalogo_perfiles.csv", index=False)
matriz_zona_perfil.to_csv("matriz_zona_perfil.csv", index=False)
zone_profile_long.to_csv("perfil_zona_largo.csv", index=False)
zone_profile_summary.to_csv("perfil_zona_resumen.csv", index=False)

ranking_prioridad = zone_profile_long.sort_values(["zone", "priority_rank"]).copy()
ranking_prioridad.to_csv("ranking_zona_perfil.csv", index=False)

columnas_pred = ["zone", "crime_profile", "month", "target_month", "target_next_month",
                 "lag_0"] + list(preds.keys())
test_eval[columnas_pred].to_csv("predicciones_test.csv", index=False)

pred_export = (
    test_eval.groupby(["zone", "crime_profile"])
    .agg(
        observado_medio=("target_next_month", "mean"),
        predicho_medio=("MLP", "mean"),
        persistencia_media=("Persistencia", "mean"),
        n_meses=("target_next_month", "size"),
    ).reset_index()
)
pred_export.to_csv("predicciones_zona_perfil.csv", index=False)

model.save("mlp_perfilamiento.keras")
print("Exportados:")
print("  auditoria_tipificaciones.csv")
print("  catalogo_perfiles.csv")
print("  matriz_zona_perfil.csv")
print("  perfil_zona_largo.csv")
print("  perfil_zona_resumen.csv")
print("  ranking_zona_perfil.csv")
print("  resultados_modelos.csv")
print("  predicciones_test.csv")
print("  predicciones_zona_perfil.csv")
print("  mlp_perfilamiento.keras")

print("Versiones -> TensorFlow", tf.__version__, "| pandas", pd.__version__,
      "| scikit-learn", __import__("sklearn").__version__)
_ = gc.collect()


## 17. Prototipo funcional

El prototipo mínimo recibe una **zona**, un **perfil** y un **mes objetivo**, recupera las features históricas de esa combinación y devuelve el conteo estimado por la MLP. Es una inferencia sobre combinaciones con historial suficiente, no un pronóstico en vivo de cualquier zona nueva.

También se expone una consulta de prioridad por zona: perfil principal, perfil prioritario y score. Así se conecta el pronóstico con la decisión de negocio sin depender de un dashboard externo.


In [ ]:

def construir_features(zona, perfil, mes_objetivo):
    fila = model_data[
        (model_data["zone"] == zona)
        & (model_data["crime_profile"] == perfil)
        & (model_data["target_month"] == mes_objetivo)
    ]
    if fila.empty:
        raise ValueError("No hay historial suficiente para esa combinación zona-perfil-mes.")
    fila = fila.iloc[[0]]
    num = num_scaler.transform(fila[NUM_FEATURES])
    cat = cat_encoder.transform(fila[[CAT_FEATURE]])
    X = np.hstack([num, cat]).astype("float32")
    return fila, X


def predecir_incidentes(zona, perfil, mes_objetivo):
    _, X = construir_features(zona, perfil, mes_objetivo)
    pred = np.expm1(np.clip(model.predict(X, verbose=0).ravel(), 0, LOG_CAP))[0]
    return float(pred)


def recomendar_prioridad(zona):
    fila = zone_profile_summary[zone_profile_summary["zone"] == zona]
    if fila.empty:
        raise ValueError("Zona sin perfil calculado.")
    r = fila.iloc[0]
    return {
        "zona": zona,
        "perfil_principal": r["perfil_principal"],
        "perfil_prioritario": r["perfil_prioritario"],
        "prioridad_score": round(float(r["prioridad_score"]), 3),
        "zona_mixta": bool(r["zona_mixta"]),
        "share_principal": round(float(r["perfil_principal_share"]), 3),
    }


ejemplos = test[["zone", "crime_profile", "target_month"]].drop_duplicates().head(5)
for _, r in ejemplos.iterrows():
    pred = predecir_incidentes(r["zone"], r["crime_profile"], r["target_month"])
    print(f"Zona {r['zone']} | {r['crime_profile']} | {r['target_month']} -> {pred:.2f} incidentes")

print("\nPrioridad de las primeras zonas por score:")
top_priority_zones = zone_profile_summary.sort_values("prioridad_score", ascending=False).head(5)
display(pd.DataFrame([recomendar_prioridad(z) for z in top_priority_zones["zone"]]))


## 18. Hotspots y perfiles (complementario)

Esta sección no reemplaza el análisis espacial de la Unidad 2; aquí se agrega una lectura rápida y reproducible para conectar la matriz zona × perfil con la ubicación de los incidentes.

Se agrupan los incidentes con coordenadas válidas en celdas de aproximadamente 500 m y se conservan las celdas con volumen suficiente. Cada **hotspot de celda** se caracteriza por volumen, perfil principal, perfiles secundarios, zona dominante, tipo de lugar, horario, fin de semana y uso de arma. Es una caracterización exploratoria, no un modelo de riesgo espacial.

In [ ]:
if int(D["valid_geo"].sum()) < 1000:
    print("Coordenadas válidas insuficientes para caracterizar hotspots.")
else:
    spatial = D[D["valid_geo"] & D["crime_profile"].isin(PROFILE_CODES)].copy()
    CELL_SIZE = 0.005  # ~500 m en Los Ángeles
    spatial["lat_grid"] = (spatial["latitude"] / CELL_SIZE).round() * CELL_SIZE
    spatial["lon_grid"] = (spatial["longitude"] / CELL_SIZE).round() * CELL_SIZE

    cell = (
        spatial.groupby(["lat_grid", "lon_grid"])
        .agg(
            incidentes=("crime_profile", "size"),
            night_share=("night", "mean"),
            weekend_share=("weekend", "mean"),
            weapon_share=("has_weapon", "mean"),
        ).reset_index()
    )
    profile_counts = (
        spatial.groupby(["lat_grid", "lon_grid", "crime_profile"]).size()
        .unstack(fill_value=0).reindex(columns=PROFILE_CODES, fill_value=0).reset_index()
    )
    cell = cell.merge(profile_counts, on=["lat_grid", "lon_grid"], how="left")
    cell = cell[cell["incidentes"] >= 30].copy()

    if len(cell) < 10:
        print("Celdas con volumen suficiente insuficientes para caracterizar hotspots.")
    else:
        profile_matrix = cell[PROFILE_CODES].to_numpy(dtype=float)
        totals = profile_matrix.sum(axis=1)
        shares = np.divide(profile_matrix, totals[:, None], out=np.zeros_like(profile_matrix), where=totals[:, None] > 0)
        main_idx = shares.argmax(axis=1)
        secondary = np.argsort(-shares, axis=1)[:, 1:4]
        hotspot_summary = pd.DataFrame({
            "hotspot_id": np.arange(1, len(cell) + 1),
            "lat_grid": cell["lat_grid"].values,
            "lon_grid": cell["lon_grid"].values,
            "incidentes": cell["incidentes"].values,
            "perfil_principal": [PROFILE_CODES[i] for i in main_idx],
            "share_principal": shares[np.arange(len(cell)), main_idx],
            "perfil_secundario_1": [PROFILE_CODES[i] for i in secondary[:, 0]],
            "perfil_secundario_2": [PROFILE_CODES[i] for i in secondary[:, 1]],
            "perfil_secundario_3": [PROFILE_CODES[i] for i in secondary[:, 2]],
            "night_share": cell["night_share"].values,
            "weekend_share": cell["weekend_share"].values,
            "weapon_share": cell["weapon_share"].values,
        })
        hotspot_summary["perfil_principal_nombre"] = hotspot_summary["perfil_principal"].map(PROFILE_LABELS)

        cell_map = cell[["lat_grid", "lon_grid"]].copy()
        cell_map["hotspot_id"] = np.arange(1, len(cell) + 1)
        spatial_h = spatial.merge(cell_map, on=["lat_grid", "lon_grid"], how="inner")
        zone_mode = (
            spatial_h.groupby(["hotspot_id", "zone"]).size().rename("n").reset_index()
            .sort_values(["hotspot_id", "n"], ascending=[True, False])
            .drop_duplicates("hotspot_id")[["hotspot_id", "zone"]]
            .rename(columns={"zone": "zona_principal"})
        )
        premise_mode = (
            spatial_h.groupby(["hotspot_id", "premise_group"]).size().rename("n").reset_index()
            .sort_values(["hotspot_id", "n"], ascending=[True, False])
            .drop_duplicates("hotspot_id")[["hotspot_id", "premise_group"]]
            .rename(columns={"premise_group": "premise_top"})
        )
        hotspot_summary = (
            hotspot_summary.merge(zone_mode, on="hotspot_id", how="left")
            .merge(premise_mode, on="hotspot_id", how="left")
        )
        hotspot_summary = hotspot_summary.sort_values("incidentes", ascending=False).reset_index(drop=True)
        hotspot_summary["rank"] = np.arange(1, len(hotspot_summary) + 1)
        hotspot_summary.to_csv("hotspots_perfil.csv", index=False)
        print(f"Celdas-hotspot con >= 30 incidentes: {len(hotspot_summary):,}")
        display(hotspot_summary.head(20).round(3))
        print("Nota: celdas de ~500 m; el análisis espacial detallado se conserva en la Unidad 2.")

### Dependencias y reproducción

En Colab o en un entorno limpio:

```bash
pip install tensorflow-cpu pandas scikit-learn matplotlib seaborn
```

En JupyterLab, el kernel debe tener TensorFlow instalado. El notebook detecta el CSV en el directorio actual, acepta `CRIME_DATA` con una ruta explícita, `FILE_NAME` con un nombre de archivo, o `CRIME_NROWS` para una prueba rápida. Para una prueba de humo se recomienda usar al menos 700.000 filas: con menos historia, los rezagos de 12 meses y la división temporal pueden quedar sin meses suficientes.

**Orden de ejecución:** todas las celdas de arriba hacia abajo. Las celdas de entrenamiento pueden tardar varios minutos en CPU. Los resultados exportados son relativos al directorio del notebook.

**Salidas principales:** `catalogo_perfiles.csv`, `auditoria_tipificaciones.csv`, `matriz_zona_perfil.csv`, `perfil_zona_resumen.csv`, `perfil_zona_largo.csv`, `ranking_zona_perfil.csv`, `resultados_modelos.csv`, `predicciones_test.csv`, `predicciones_zona_perfil.csv`, `hotspots_perfil.csv` y `mlp_perfilamiento.keras`. Estos archivos alimentan el informe y el prototipo analítico; el dashboard web no está conectado.